# Figure 6: lung cancer atlas and clinical associations

Use saved Salcher concepts to annotate the lung cancer atlas, compare concept activity across TCGA tumor stages, assess patient survival, and visualize SCISSOR associations. Run the sections in order after configuring paths and analysis switches. Outputs are saved to `outputs/figure6`.

## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.

Survival requires `lifelines`. Drug-ranking Excel inputs require `openpyxl`, and the interactive Sankey requires `plotly`; optional PDF export additionally requires a working `kaleido` backend.


In [ ]:
from pathlib import Path
import os
import re
import json
import hashlib
import platform
import importlib.metadata
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from IPython.display import display

# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
CONCEPT_PATH = CONCEPT_DIR / "GPT5_Salcher_concepts.json"
ATLAS_PATH = input_path("lung.atlas", "Salcher.h5ad")
TCGA_PATH = input_path("lung.tcga", "TCGA_lung.h5ad")
SCISSOR_LABEL_PATH = input_path("lung.scissor_labels", "Scissor_all_cells_labels.tsv")
GMT_PATH = input_path("gmt.Hs", "msigdb.v2024.1.Hs.symbols.gmt")
OUTPUT_DIR = TUTORIAL_DIR / "outputs/figure6"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ATLAS = True
RUN_ATLAS_PLOTS = True
RUN_INTERPRETABILITY = True
RUN_TCGA = True
RUN_STAGE_PLOTS = True
RUN_SURVIVAL = True
RUN_SCISSOR = True
ATLAS_TOP_K = 100
TCGA_TOP_K = 10
SCISSOR_TOP_K = 10
FOCUS = ["B cell program", "Mitotic proliferation"]
concepts = load_saved_concepts(CONCEPT_PATH)
concept_names = [c["name"] for c in concepts]
assert set(FOCUS).issubset(concept_names)


## 1. Concept scoring and metrics

For each gene, standardize expression across the complete input cohort. Compute concept activity as the mean standardized expression of matched genes, retaining their listed order. Use 100 genes for atlas annotation and 10 genes for TCGA and SCISSOR concept mapping.

Report ARI, NMI, and AMI separately. TC uses the first 10 genes and human MSigDB pathway co-occurrence. TD counts gene entries occurring exactly once; `TD_unique_fraction` also reports the distinct-symbol fraction.

The helpers below retain the reported TC/TD definitions. They are evaluated from each current concept set, rather than reusing cached annotations from `ScConcept.evaluation()`.


In [ ]:
def score_concepts(data, definitions, top_k):
    gene_index = {str(g): i for i, g in enumerate(data.var_names)}
    scores = np.zeros((data.n_obs, len(definitions)), dtype=float)
    matches = []
    for i, concept in enumerate(definitions):
        selected = concept["genes"][:top_k]
        idx = [gene_index[g] for g in selected if g in gene_index]
        if not idx:
            raise ValueError(f"No matched genes for {concept['name']!r}.")
        # Backed sparse inputs require increasing indices; restore the requested order afterward.
        unique_idx = sorted(set(idx))
        block = data[:, unique_idx].X
        block = block.toarray() if sparse.issparse(block) else np.asarray(block)
        block = block.astype(float)
        if not np.isfinite(block).all():
            raise ValueError("Non-finite concept-gene expression values.")
        block = (block - block.mean(axis=0, keepdims=True)) / (block.std(axis=0, keepdims=True) + 1e-6)
        position = {col: j for j, col in enumerate(unique_idx)}
        scores[:, i] = block[:, [position[j] for j in idx]].mean(axis=1)
        matches.append(dict(concept=concept["name"], n_requested=len(selected), n_matched=len(idx),
                            missing_genes=[g for g in selected if g not in gene_index]))
    return scores, matches


## 2. Annotate the lung cancer atlas

Load the atlas and assign each cell to its highest-scoring concept. Reuse stored UMAP coordinates when available. If coordinates are absent, calculate PCA, neighbors, and UMAP with default Scanpy settings. Save cell-level annotations, scores, gene matches, and metrics.


In [ ]:
atlas_out = OUTPUT_DIR / "atlas"
atlas_out.mkdir(exist_ok=True)
if RUN_ATLAS:
    atlas = sc.read_h5ad(ATLAS_PATH, backed="r")
    atlas_scores, atlas_gene_matches = score_concepts(atlas, concepts, ATLAS_TOP_K)
    atlas_assignments = np.asarray(concept_names)[atlas_scores.argmax(axis=1)]
    truth = atlas.obs["cell_type_coarse"].astype(str).to_numpy()
    atlas_metrics = dict(n_cells=atlas.n_obs, n_concepts=len(concepts),
        ARI=adjusted_rand_score(truth, atlas_assignments),
        NMI=normalized_mutual_info_score(truth, atlas_assignments),
        AMI=adjusted_mutual_info_score(truth, atlas_assignments))
    if RUN_INTERPRETABILITY:
        atlas_metrics.update(interpretability(concepts, list(atlas.var_names), read_pathways(GMT_PATH), 10))
    pd.DataFrame([atlas_metrics]).to_csv(atlas_out / "scConcept_metrics.csv", index=False)
    pd.DataFrame({"cell_id": atlas.obs_names, "cell_type": truth, "concept": atlas_assignments}).to_csv(
        atlas_out / "cell_assignments.csv", index=False)
    np.savez_compressed(atlas_out / "concept_scores.npz", scores=atlas_scores, names=np.asarray(concept_names))
    (atlas_out / "gene_matches.json").write_text(json.dumps(atlas_gene_matches, indent=2))
    display(pd.DataFrame([atlas_metrics]))
    if RUN_ATLAS_PLOTS:
        atlas_view = ad.AnnData(np.zeros((atlas.n_obs, 1), dtype=np.float32), obs=atlas.obs.copy())
        atlas_view.obs["concept"] = pd.Categorical(atlas_assignments)
        if "X_umap" in atlas.obsm:
            atlas_view.obsm["X_umap"] = np.asarray(atlas.obsm["X_umap"]).copy()
        else:
            embedding_data = atlas.to_memory()
            sc.tl.pca(embedding_data)
            sc.pp.neighbors(embedding_data)
            sc.tl.umap(embedding_data)
            atlas_view.obsm["X_umap"] = embedding_data.obsm["X_umap"].copy()
            del embedding_data
        for key, name in [("cell_type_coarse", "Fig6A_cell_types.pdf"), ("concept", "Fig6B_concepts.pdf")]:
            fig = sc.pl.umap(atlas_view, color=key, frameon=False, legend_loc="right margin", return_fig=True, show=False)
            fig.savefig(atlas_out / name, bbox_inches="tight")
            plt.show()
            plt.close(fig)
        np.savez_compressed(atlas_out / "expression_umap.npz", coordinates=atlas_view.obsm["X_umap"],
                            cell_ids=atlas_view.obs_names.to_numpy(dtype=str))
    atlas.file.close()


## 3. Transfer concepts to TCGA

Load TCGA lung expression and clinical annotations. Use the first 10 listed genes per concept and gene-wise standardization across all patients. Keep sample identifiers aligned with scores.


In [ ]:
clinical_out = OUTPUT_DIR / "TCGA"
clinical_out.mkdir(exist_ok=True)
if RUN_TCGA:
    TCGA_lung = sc.read_h5ad(TCGA_PATH)
    TCGA_lung.obs_names = TCGA_lung.obs["TCGA_patient_barcode"].astype(str)
    if not TCGA_lung.obs_names.is_unique:
        raise ValueError("TCGA patient identifiers must be unique.")
    scores, tcga_gene_matches = score_concepts(TCGA_lung, concepts, TCGA_TOP_K)
    scores_df = pd.DataFrame(scores, columns=concept_names)
    meta = TCGA_lung.obs[["TCGA_patient_barcode", "tumor_stage", "tumor_stage_ajcc", "type", "time", "status", "age"]].copy().reset_index(drop=True)
    df = pd.concat([meta, scores_df], axis=1)
    df.to_csv(clinical_out / "patient_concept_scores.csv", index=False)
    (clinical_out / "gene_matches.json").write_text(json.dumps(tcga_gene_matches, indent=2))
    print("TCGA patients:", len(df))


In [ ]:
mpl.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12, 'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11})

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import matplotlib as mpl
import re

def sanitize_filename(name):
    """
    Make a string safe for use as a filename.
    """
    name = name.replace(' ', '_')
    name = name.replace('/', '_')
    name = name.replace('+', 'plus')
    name = re.sub('[^A-Za-z0-9_.-]', '', name)
    return name

def plot_concept_vs_tumor_stage(df, score_col, stage_col='tumor_stage', tumor_col='type', tumor_type=None, bad_stage_val=-2147483648, stage_labels=('Early stage', 'Advanced stage'), figsize=(4.2, 3.6), out_dir=None, save=True):
    """
    Nature-style boxplot comparing concept scores between early vs advanced stage tumors.
    Uses Mann–Whitney U test (two-sided).
    """
    if out_dir is None:
        out_dir = './'
    os.makedirs(out_dir, exist_ok=True)
    cols = [score_col, stage_col]
    if tumor_col is not None and tumor_col in df.columns:
        cols.append(tumor_col)
    d = df[cols].copy()
    if tumor_type is not None:
        d = d[d[tumor_col] == tumor_type].copy()
    d.loc[d[stage_col] == bad_stage_val, stage_col] = np.nan
    d = d.dropna(subset=[score_col, stage_col])
    d = d[d[stage_col].isin([0, 1])].copy()
    x0 = d.loc[d[stage_col] == 0, score_col].values
    x1 = d.loc[d[stage_col] == 1, score_col].values
    (n0, n1) = (len(x0), len(x1))
    if n0 < 2 or n1 < 2:
        p = np.nan
    else:
        (_, p) = mannwhitneyu(x0, x1, alternative='two-sided')
    (fig, ax) = plt.subplots(figsize=figsize)
    colors = ['#4C72B0', '#DD8452']
    bp = ax.boxplot([x0, x1], labels=list(stage_labels), widths=0.5, showfliers=False, patch_artist=True)
    for (patch, color) in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.0)
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1.0)
    for whisker in bp['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(1.0)
    for cap in bp['caps']:
        cap.set_color('black')
        cap.set_linewidth(1.0)
    if np.isnan(p):
        p_str = 'p = NA'
    else:
        p_str = f'$p = {p:.3g}$'
    ax.set_title(f'{score_col}\n{p_str}, $n_{{Early}}={n0}$, $n_{{Advanced}}={n1}$', fontsize=11)
    ax.set_ylabel(f'{score_col} score')
    ax.grid(axis='y', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    if save:
        safe_name = sanitize_filename(score_col)
        out_path = os.path.join(out_dir, f'{safe_name}_stage_{tumor_type or "NSCLC"}.pdf')
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved: {out_path}')
    plt.show()
    plt.close()
    return p


## 4. Compare early and advanced tumor stages

Compare B-cell and mitotic-proliferation concept scores using two-sided Mann–Whitney U tests. `tumor_stage` encodes early stage as 0 and advanced stage as 1. Save the stage boxplots and P values.


In [ ]:
if RUN_TCGA and RUN_STAGE_PLOTS:
    stage_results = []
    for concept in FOCUS:
        pvalue = plot_concept_vs_tumor_stage(df, concept, tumor_type=None, out_dir=str(clinical_out))
        stage_results.append(dict(concept=concept, mannwhitney_p=pvalue))
    pd.DataFrame(stage_results).to_csv(clinical_out / "stage_tests.csv", index=False)


## 5. Assess patient survival

Compare the bottom and top 25% of concept scores with Kaplan–Meier curves and log-rank tests. Convert follow-up time from days to months for plotting. Fit continuous-score Cox models, reporting hazard ratios per standard deviation, adjusted for age and early/advanced stage; the combined cohort also includes LUAD/LUSC subtype. Analyze NSCLC, LUAD, and LUSC separately.


In [ ]:
if RUN_TCGA and RUN_SURVIVAL:
    import numpy as np
    import pandas as pd
    from lifelines import CoxPHFitter
    scores_patient = pd.DataFrame(scores, columns=concept_names)
    scores_patient['TCGA_patient_barcode'] = TCGA_lung.obs['TCGA_patient_barcode'].astype(str).values
    clinical = TCGA_lung.obs[['TCGA_patient_barcode', 'time', 'status', 'age', 'tumor_stage', 'type']].copy().reset_index(drop=True)
    clinical['TCGA_patient_barcode'] = clinical['TCGA_patient_barcode'].astype(str)
    for c in ['time', 'status', 'age', 'tumor_stage']:
        clinical[c] = pd.to_numeric(clinical[c], errors='coerce')
    clinical.loc[clinical['time'] <= 0, 'time'] = np.nan
    clinical.loc[clinical['age'] < 0, 'age'] = np.nan
    clinical.loc[~clinical['tumor_stage'].isin([0, 1]), 'tumor_stage'] = np.nan
    clinical = clinical[clinical['status'].isin([0, 1])].copy()
    cox_df = clinical.merge(scores_patient, on='TCGA_patient_barcode', how='inner', validate='one_to_one')
    print('Merged N =', len(cox_df))


In [ ]:
if RUN_TCGA and RUN_SURVIVAL:
    import os
    import re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from lifelines import KaplanMeierFitter, CoxPHFitter
    from lifelines.statistics import logrank_test
    OUT_DIR = str(clinical_out)
    os.makedirs(OUT_DIR, exist_ok=True)
    FOCUS = ['B cell program', 'Mitotic proliferation']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13, 'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10, 'axes.linewidth': 1.0, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    
    def safe_name(x):
        return re.sub('[^A-Za-z0-9_.-]', '', x.replace(' ', '_'))
    
    def plot_km_with_adjusted_cox(df, concept, tumor_type=None, save=True):
        d = df.copy()
        if tumor_type is not None:
            d = d[d['type'] == tumor_type].copy()
        d = d.dropna(subset=['time', 'status', concept]).copy()
        d = d[(d['time'] > 0) & d['status'].isin([0, 1])].copy()
        d['time_month'] = d['time'] / 30.0
        q25 = d[concept].quantile(0.25)
        q75 = d[concept].quantile(0.75)
        low = d[d[concept] <= q25].copy()
        high = d[d[concept] >= q75].copy()
        lr = logrank_test(high['time_month'], low['time_month'], event_observed_A=high['status'], event_observed_B=low['status'])
        p_logrank = float(lr.p_value)
        cox_cols = ['time', 'status', concept, 'tumor_stage', 'age']
        if tumor_type is None:
            cox_cols.append('type')
        dc = d[cox_cols].dropna().copy()
        dc = dc[dc['tumor_stage'].isin([0, 1]) & (dc['age'] >= 0)].copy()
        dc['concept_z'] = (dc[concept] - dc[concept].mean()) / dc[concept].std(ddof=1)
        fit_cols = ['time', 'status', 'concept_z', 'tumor_stage', 'age']
        if tumor_type is None:
            dc['LUSC'] = (dc['type'] == 'LUSC').astype(int)
            fit_cols.append('LUSC')
        cph = CoxPHFitter()
        cph.fit(dc[fit_cols], duration_col='time', event_col='status')
        s = cph.summary.loc['concept_z']
        HR = float(np.exp(s['coef']))
        CI_low = float(np.exp(s['coef lower 95%']))
        CI_high = float(np.exp(s['coef upper 95%']))
        p_cox = float(s['p'])
        (fig, ax) = plt.subplots(figsize=(4.6, 3.8))
        COLOR_HIGH = '#E76F51'
        COLOR_LOW = '#6EC0C2'
        kmf = KaplanMeierFitter()
        kmf.fit(high['time_month'], high['status'], label=f'High (top 25%, n={len(high)})')
        kmf.plot_survival_function(ax=ax, ci_show=False, linewidth=2.2, color=COLOR_HIGH)
        kmf.fit(low['time_month'], low['status'], label=f'Low (bottom 25%, n={len(low)})')
        kmf.plot_survival_function(ax=ax, ci_show=False, linewidth=2.2, color=COLOR_LOW)
        text = f'Log-rank $P$ = {p_logrank:.3g}\nAdjusted Cox HR = {HR:.2f}\n95% CI: {CI_low:.2f}–{CI_high:.2f}\nCox $P$ = {p_cox:.3g}'
        ax.text(0.05, 0.07, text, transform=ax.transAxes, va='bottom', fontsize=10)
        title = concept if tumor_type is None else f'{tumor_type} — {concept}'
        ax.set_title(title)
        ax.set_xlabel('Time (months)')
        ax.set_ylabel('Survival probability')
        ax.set_ylim(0, 1.02)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False, loc='upper right')
        plt.tight_layout()
        if save:
            cohort = 'NSCLC' if tumor_type is None else tumor_type
            path = os.path.join(OUT_DIR, f'{cohort}_{safe_name(concept)}_KM_Cox.pdf')
            plt.savefig(path, dpi=300, bbox_inches='tight')
            print('Saved:', path)
        plt.show()
        plt.close()
        return {'tumor': 'NSCLC' if tumor_type is None else tumor_type, 'concept': concept, 'KM_high_n': len(high), 'KM_low_n': len(low), 'logrank_P': p_logrank, 'Cox_N': len(dc), 'Cox_events': int(dc['status'].sum()), 'HR_adjusted': HR, 'CI_low': CI_low, 'CI_high': CI_high, 'Cox_P': p_cox}
    plot_results = []
    for concept in FOCUS:
        r = plot_km_with_adjusted_cox(cox_df, concept, tumor_type=None)
        plot_results.append(r)
    for tumor in ['LUAD', 'LUSC']:
        for concept in FOCUS:
            r = plot_km_with_adjusted_cox(cox_df, concept, tumor_type=tumor)
            plot_results.append(r)
    survival_plot_results = pd.DataFrame(plot_results)
    display(survival_plot_results.round(4))
    survival_plot_results.to_csv(clinical_out / "survival_results.csv", index=False)


## 6. Map SCISSOR survival associations

Read cell-level SCISSOR labels and map them by cell ID to primary-tumor cells. Compute concept scores across the full atlas before selecting primary tumors. For each patient/concept group with more than 10 cells, use a pseudocount of 0.01 and compare positive/negative fractions with paired Wilcoxon tests and Benjamini–Hochberg correction.

The display maps `Scissor-` to better survival (blue) and `Scissor+` to worse survival (red). The concept bar plot displays log2(negative/positive) fraction ratios with FDR < 0.05. Missing SCISSOR labels are shown in gray.


In [ ]:
scissor_out = OUTPUT_DIR / "SCISSOR"
scissor_out.mkdir(exist_ok=True)
if RUN_SCISSOR:
    scissor_atlas = sc.read_h5ad(ATLAS_PATH, backed="r")
    scissor_scores, scissor_gene_matches = score_concepts(scissor_atlas, concepts, SCISSOR_TOP_K)
    primary = scissor_atlas.obs["origin"].astype(str).to_numpy() == "tumor_primary"
    adata_plot = scissor_atlas[primary].to_memory()
    adata_plot.obs["concept"] = pd.Categorical(np.asarray(concept_names)[scissor_scores[primary].argmax(axis=1)])
    scissor_atlas.file.close()
    label_table = pd.read_csv(SCISSOR_LABEL_PATH, sep="\t", dtype=str, keep_default_na=True, na_values=["", "NA", "NaN"])
    label_table = label_table[["cell_id", "scissor_label"]].drop_duplicates("cell_id", keep="first")
    mapping = pd.Series(label_table.scissor_label.values, index=label_table.cell_id)
    adata_plot.obs["scissor"] = pd.Index(adata_plot.obs_names).map(mapping)
    adata_plot.obs.to_csv(scissor_out / "cell_annotations.csv")
    (scissor_out / "gene_matches.json").write_text(json.dumps(scissor_gene_matches, indent=2))


In [ ]:
if RUN_SCISSOR:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    from scipy.stats import wilcoxon
    from statsmodels.stats.multitest import multipletests
    import os
    PSEUDOCOUNT = 0.01
    MIN_CELLS_PER_PATIENT_CONCEPT = 10
    FDR_CUTOFF = 0.05
    OUT_FIG_DIR = str(scissor_out)
    os.makedirs(OUT_FIG_DIR, exist_ok=True)
    OUT_FIG = os.path.join(OUT_FIG_DIR, 'TCGA_Scissor_sur_log.pdf')
    df = adata_plot.obs[['patient', 'concept', 'scissor']].copy()
    df['scissor_group'] = df['scissor'].astype(str)
    df['scissor_group'] = df['scissor_group'].where(df['scissor_group'].isin(['Scissor+', 'Scissor-']), other='neutral')
    df = df.dropna(subset=['patient', 'concept']).copy()
    cnt = df.groupby(['patient', 'concept', 'scissor_group']).size().unstack(fill_value=0)
    for col in ['Scissor+', 'Scissor-', 'neutral']:
        if col not in cnt.columns:
            cnt[col] = 0
    cnt['total'] = cnt['Scissor+'] + cnt['Scissor-'] + cnt['neutral']
    cnt = cnt[cnt['total'] > MIN_CELLS_PER_PATIENT_CONCEPT].copy()
    cnt_pc = cnt[['Scissor+', 'Scissor-', 'neutral']].astype(float) + PSEUDOCOUNT
    frac = cnt_pc.div(cnt_pc.sum(axis=1), axis=0)
    results = []
    concepts = sorted(frac.index.get_level_values('concept').unique())
    for c in concepts:
        sub = frac.xs(c, level='concept')
        x = sub['Scissor+'].values
        y = sub['Scissor-'].values
        mean_pos = float(np.mean(x))
        mean_neg = float(np.mean(y))
        log2_ratio = float(np.log2(mean_pos / mean_neg))
        try:
            (_, pval) = wilcoxon(x, y, zero_method='zsplit', alternative='two-sided')
        except ValueError:
            pval = np.nan
        results.append({'concept': c, 'n_patients_used': sub.shape[0], 'log2_ratio': log2_ratio, 'pval': pval})
    res = pd.DataFrame(results)
    mask = res['pval'].notna()
    res.loc[mask, 'fdr'] = multipletests(res.loc[mask, 'pval'], method='fdr_bh')[1]
    res.loc[~mask, 'fdr'] = np.nan
    res_sig = res[res['fdr'] < FDR_CUTOFF].copy()
    res_sig = res_sig.sort_values('log2_ratio').reset_index(drop=True)
    print('✅ total concepts:', res.shape[0])
    print(f'✅ significant (BH-FDR < {FDR_CUTOFF}):', res_sig.shape[0])
    if res_sig.shape[0] == 0:
        print('⚠️ No significant concepts. Nothing to plot.')
    else:
        res_sig['log2_ratio'] = -res_sig['log2_ratio']
        res_sig = res_sig.iloc[::-1].reset_index(drop=True)
        vals = res_sig['log2_ratio'].values
        vmax = np.max(np.abs(vals))
        cmap = plt.cm.RdBu
        norm = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        colors = cmap(norm(vals))
        (fig, ax) = plt.subplots(figsize=(9.5, 4.2))
        ax.bar(res_sig['concept'], res_sig['log2_ratio'], color=colors, edgecolor='none')
        ax.axhline(0, color='#9e9e9e', linewidth=1.0, zorder=0)
        ax.tick_params(axis='x', rotation=45)
        plt.setp(ax.get_xticklabels(), ha='right', fontsize=16)
        ax.grid(axis='y', alpha=0.15)
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, location='left', fraction=0.1, pad=0.04)
        cbar.ax.tick_params(labelsize=11)
        cbar.set_label('Log2-ratio', fontsize=16)
        for spine in ax.spines.values():
            spine.set_color('#9e9e9e')
            spine.set_linewidth(1.0)
        cbar.ax.yaxis.set_label_position('left')
        cbar.ax.yaxis.set_ticks_position('left')
        ax.set_yticks([])
        plt.tight_layout()
        plt.savefig(OUT_FIG, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'✅ Figure saved to: {OUT_FIG}')
    res_sig.head()
    res.to_csv(scissor_out / "concept_survival_statistics.csv", index=False)


In [ ]:
if RUN_SCISSOR:
    adata_plot.obs["scissor_plot"] = pd.Categorical(
        adata_plot.obs["scissor"].astype("object").fillna("NA"),
        categories=["Scissor+", "Scissor-", "NA"], ordered=True)
    sc.pp.pca(adata_plot)
    sc.pp.neighbors(adata_plot)
    sc.tl.umap(adata_plot)
    OUT_FIG = str(scissor_out / "Fig6H_survival_scissor.pdf")
    OUT_FIG_DIR = str(scissor_out)


In [ ]:
if RUN_SCISSOR:
    (fig, ax) = plt.subplots(figsize=(9, 7))
    adata_plot.obs['survival_plot'] = adata_plot.obs['scissor_plot'].map({'Scissor-': 'Better survival', 'Scissor+': 'Worse survival', 'NA': 'NA'}).astype(str)
    sc.pl.umap(adata_plot[adata_plot.obs['survival_plot'] == 'NA'], color='survival_plot', palette={'NA': '#BDBDBD'}, size=2, alpha=0.1, frameon=False, legend_loc=None, title='', ax=ax, show=False)
    sc.pl.umap(adata_plot[adata_plot.obs['survival_plot'].isin(['Better survival', 'Worse survival'])], color='survival_plot', palette={'Better survival': '#1f77b4', 'Worse survival': '#d62728'}, size=9, alpha=0.8, frameon=False, legend_loc=None, title='', ax=ax, show=False)
    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=16, label='Better survival'), Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markersize=16, label='Worse survival'), Line2D([0], [0], marker='o', color='w', markerfacecolor='#BDBDBD', markersize=16, alpha=0.6, label='NA')]
    ax.legend(handles=handles, labels=[h.get_label() for h in handles], loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False, fontsize=18, handletextpad=0.4, columnspacing=1.2)
    plt.tight_layout()
    plt.savefig(OUT_FIG, dpi=300, bbox_inches='tight')
    OUT_FIG = os.path.join(OUT_FIG_DIR, 'TCGA_UMAP_survival_scissor.png')
    plt.show()
    plt.close()


## 7. Rank candidate drugs

Read saved drug–protein affinity predictions, approved-drug names/SMILES, protein sequences, and the TTD lung-cancer drug list from `Inputs/drug_ranking`.

Map ligand SMILES to drug names and keep the maximum score for duplicate drug–protein pairs. Select KIF20A, AURKB, NEK2, and PBK by their protein sequences. Rank each drug by its maximum affinity across the four targets and print the top 10. This section uses saved predictions and does not rerun affinity prediction.


In [ ]:
DRUG_INPUT_DIR = TUTORIAL_DIR / "Inputs/drug_ranking"
DRUG_OUTPUT_DIR = OUTPUT_DIR / "drug_ranking"
DRUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DRUG_ANALYSIS = True
RUN_DRUG_SANKEY = True
EXPORT_SANKEY_PDF = False  # Enable when Kaleido static export is available.
TARGET_GENES = ["KIF20A", "AURKB", "NEK2", "PBK"]


def load_drug_matrix(input_dir):
    current_name = None
    rows = []
    with (input_dir / "P3-07-Approved_smi_inchi.txt").open(encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            line = line.rstrip("\r\n")
            if line.startswith("DRUGNAME\t"):
                parts = line.split("\t", 1)
                if len(parts) == 2 and parts[1].strip():
                    current_name = parts[1].strip()
            elif line.startswith("DRUGSMIL\t"):
                parts = line.split("\t", 1)
                smiles = parts[1].strip() if len(parts) == 2 else ""
                if current_name is not None and smiles:
                    rows.append(dict(drug_name=current_name, smiles=smiles))
    drugmap = pd.DataFrame(rows).drop_duplicates().dropna()
    drugmap = drugmap.loc[~(drugmap.drug_name.str.lower().isin(["drug name"]) |
                            drugmap.smiles.str.lower().isin(["canonical smiles"]))]
    smiles_to_name = drugmap.drop_duplicates("smiles").set_index("smiles")["drug_name"].to_dict()
    screen = pd.read_csv(input_dir / "Tra_lung_screening.csv")
    required = {"Protein", "Ligand", "predicted_binding_affinity"}
    if not required.issubset(screen.columns):
        raise ValueError(f"Missing prediction columns: {required - set(screen.columns)}")
    for column in ["Protein", "Ligand"]:
        screen[column] = screen[column].astype(str).str.strip()
    screen["predicted_binding_affinity"] = pd.to_numeric(screen["predicted_binding_affinity"], errors="coerce")
    screen = screen.dropna(subset=["predicted_binding_affinity"])
    screen["drug_name"] = screen.Ligand.map(smiles_to_name)
    screen = screen.dropna(subset=["drug_name"]).copy()
    screen["drug_name"] = screen["drug_name"].astype(str).str.strip()
    proteins = pd.read_excel(input_dir / "gene2pro_lung.xlsx", header=None)
    sequence_map = {str(row.iloc[0]).strip(): re.sub(r"[\s\\]+", "", str(row.iloc[2]))
                    for _, row in proteins.iterrows()}
    target_sequences = [sequence_map[gene] for gene in TARGET_GENES]
    grouped = screen.groupby(["drug_name", "Protein"], as_index=False)["predicted_binding_affinity"].max()
    matrix = grouped.pivot(index="drug_name", columns="Protein", values="predicted_binding_affinity")[target_sequences].copy()
    if not np.isfinite(matrix.to_numpy()).all():
        raise ValueError("Missing or non-finite scores among the four selected targets.")
    gold = pd.read_excel(input_dir / "TTD_lung.xlsx", header=None).iloc[:, 0].astype(str).str.strip()
    gold = set(gold.loc[(gold != "") & ~gold.str.startswith("TTD", na=False)])
    target_mapping = pd.DataFrame({"gene": TARGET_GENES, "protein_sequence": target_sequences})
    return matrix, gold, target_mapping

if RUN_DRUG_ANALYSIS:
    drug_matrix, known_lung_drugs, drug_target_mapping = load_drug_matrix(DRUG_INPUT_DIR)
    drug_ranking = drug_matrix.max(axis=1).rename("score").sort_values(ascending=False).reset_index()
    drug_ranking.insert(0, "rank", np.arange(1, len(drug_ranking) + 1))
    drug_ranking["known_lung_cancer_drug"] = drug_ranking.drug_name.isin(known_lung_drugs)
    print(drug_ranking.head(10).to_string(index=False))
    drug_ranking.to_csv(DRUG_OUTPUT_DIR / "drug_ranking_max.csv", index=False)
    drug_ranking.head(10).to_csv(DRUG_OUTPUT_DIR / "top10_drugs.csv", index=False)
    drug_matrix.to_csv(DRUG_OUTPUT_DIR / "drug_target_affinities.csv")
    drug_target_mapping.to_csv(DRUG_OUTPUT_DIR / "target_mapping.csv", index=False)


### Drug-score distributions

Compare the maximum-affinity scores of known lung-cancer drugs with all other screened drugs using empirical cumulative distributions and a two-sided Kolmogorov–Smirnov test. Plot the maximum-separation marker and export the test statistics.


In [ ]:
OUT_FIG_DIR = str(DRUG_OUTPUT_DIR)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import matplotlib as mpl
mpl.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 12, 'axes.titlesize': 12, 'axes.labelsize': 12, 'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'axes.linewidth': 1.2})

def plot_cdf_with_ks(drug_scores: pd.DataFrame, y_true: np.ndarray, title_prefix: str='', pos_label: str='Known positives', neg_label: str='All other', figsize=(6, 4), lw=2.2, show_grid=True, show_ks_vline=True):
    """
    Plot empirical CDFs (0-1) for positive vs negative groups and report
    Kolmogorov–Smirnov statistic with italic p-value.

    drug_scores: DataFrame with a column named 'score'
    y_true: 1 for positives, 0 for negatives
    """
    scores_pos = drug_scores.loc[y_true == 1, 'score'].values.astype(float)
    scores_neg = drug_scores.loc[y_true == 0, 'score'].values.astype(float)
    if len(scores_pos) < 2 or len(scores_neg) < 2:
        raise ValueError('Need at least 2 samples in each group for KS test / CDF plot.')
    pos_sorted = np.sort(scores_pos)
    neg_sorted = np.sort(scores_neg)
    cdf_pos = np.arange(1, len(pos_sorted) + 1) / len(pos_sorted)
    cdf_neg = np.arange(1, len(neg_sorted) + 1) / len(neg_sorted)
    ks_stat, ks_p = ks_2samp(scores_pos, scores_neg, alternative='two-sided')
    fig, ax = plt.subplots(figsize=figsize)
    ax.step(neg_sorted, cdf_neg, where='post', linewidth=lw, label=f'{neg_label} (n={len(scores_neg)})')
    ax.step(pos_sorted, cdf_pos, where='post', linewidth=lw, label=f'{pos_label} (n={len(scores_pos)})')
    if show_ks_vline:
        xmin = min(neg_sorted.min(), pos_sorted.min())
        xmax = max(neg_sorted.max(), pos_sorted.max())
        xs = np.linspace(xmin, xmax, 800)
        cdf_neg_i = np.interp(xs, neg_sorted, cdf_neg, left=0.0, right=1.0)
        cdf_pos_i = np.interp(xs, pos_sorted, cdf_pos, left=0.0, right=1.0)
        diff = np.abs(cdf_pos_i - cdf_neg_i)
        x_ks = xs[np.argmax(diff)]
        ax.axvline(x_ks, linestyle='--', linewidth=1.2, alpha=0.6)
        ax.text(x_ks, 0.05, f'D = {ks_stat:.3f}', rotation=90, va='bottom', ha='right', fontsize=10, alpha=0.9)
    ax.set_xlabel('Ensemble score', fontsize=13)
    ax.set_ylabel('Cumulative fraction', fontsize=13)
    ax.set_ylim(0, 1.02)
    if show_grid:
        ax.grid(alpha=0.2)
    ax.set_title(f'{title_prefix}\nKolmogorov–Smirnov statistic = {ks_stat:.3f}, $p = {ks_p:.3g}$', fontsize=12, pad=10)
    ax.legend(loc='upper left', frameon=False, fontsize=12)
    plt.tight_layout()
    import os
    os.makedirs(OUT_FIG_DIR, exist_ok=True)
    safe_prefix = title_prefix.replace(' ', '_').replace('[', '').replace(']', '')
    out_pdf = os.path.join(OUT_FIG_DIR, f'{safe_prefix}.pdf')
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight', format='pdf')
    print(f'Saved: {out_pdf}')
    plt.show()
    plt.close(fig)
    return {'ks_stat': float(ks_stat), 'ks_p': float(ks_p), 'n_pos': int(len(scores_pos)), 'n_neg': int(len(scores_neg))}


In [ ]:
if RUN_DRUG_ANALYSIS:
    drug_scores = pd.DataFrame({"drug_name": drug_matrix.index.to_numpy(), "score": drug_matrix.max(axis=1).to_numpy()})
    drug_is_known = drug_scores.drug_name.isin(known_lung_drugs).astype(int).to_numpy()
    drug_ks = plot_cdf_with_ks(drug_scores, drug_is_known, title_prefix="[LUNG] method=max",
        pos_label="Known lung cancer drugs", neg_label="All other drugs", figsize=(6, 4), show_ks_vline=True)
    (DRUG_OUTPUT_DIR / "KS_test_max.json").write_text(json.dumps(drug_ks, indent=2))


### Top-10 drug–target associations

Plot the four targets and the top 10 drugs ranked by maximum affinity. Link widths use the individual drug–target scores, multiplied by 6 for display. Target labels follow the protein-sequence mapping. The interactive HTML figure requires Plotly. Set `EXPORT_SANKEY_PDF=True` to additionally export a PDF with a working Kaleido installation.


In [ ]:
import os
import numpy as np
import pandas as pd
if RUN_DRUG_ANALYSIS and RUN_DRUG_SANKEY:
    import plotly.graph_objects as go
TARGET_COLORS_HEX = ['#4E79A7', '#C44E52', '#55A868', '#DD8452']
LINK_ALPHA = 0.25
NODE_ALPHA = 0.8
LINK_MIN_VALUE = 0.0

def hex_to_rgba(hex_color: str, alpha: float) -> str:
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

def plot_target_drug_sankey_topk(mat: pd.DataFrame, cancer_type: str='LUNG', method_name: str='max', top_k_drugs: int=10, n_targets: int=4, out_dir: str=str(DRUG_OUTPUT_DIR), out_prefix=None, scale_link_values: float=1.0):
    """Plot top-ranked drugs and their four gene-labeled targets; save HTML, PDF when available, and the target mapping."""
    assert isinstance(mat, pd.DataFrame), 'mat must be a pandas DataFrame'
    assert mat.shape[1] >= n_targets, f'mat must have at least {n_targets} target columns'
    target_names = list(mat.columns[:n_targets])
    target_short = ['KIF20A', 'AURKB', 'NEK2', 'PBK']
    target_map_df = pd.DataFrame({'target_short': target_short, 'target_full': target_names})
    sub = mat[target_names].copy()
    sub = sub.apply(pd.to_numeric, errors='coerce')
    sub = sub.dropna(how='all')
    drug_max = sub.max(axis=1)
    top_drugs = drug_max.sort_values(ascending=False).head(top_k_drugs).index.tolist()
    mat_top = sub.loc[top_drugs, target_names].copy()
    node_labels = target_short + top_drugs
    node_colors = [hex_to_rgba(c, NODE_ALPHA) for c in TARGET_COLORS_HEX[:n_targets]] + ['rgba(210,210,210,0.25)'] * len(top_drugs)
    sources = []
    targets_idx = []
    values = []
    link_colors = []
    for (ti, t_full) in enumerate(target_names):
        for (di, drug) in enumerate(top_drugs):
            v = mat_top.loc[drug, t_full]
            if pd.isna(v):
                continue
            if v <= LINK_MIN_VALUE:
                continue
            sources.append(ti)
            targets_idx.append(n_targets + di)
            values.append(float(v) * scale_link_values)
            link_colors.append(hex_to_rgba(TARGET_COLORS_HEX[ti], LINK_ALPHA))
    fig = go.Figure(go.Sankey(arrangement='snap', node=dict(pad=4, thickness=15, label=node_labels, color=node_colors, line=dict(color='rgba(80,80,80,0.5)', width=0.6)), link=dict(source=sources, target=targets_idx, value=values, color=link_colors)))
    fig.update_layout(title=None, font=dict(family='Arial', size=22, color='#222222'), width=800, height=520, margin=dict(l=30, r=30, t=20, b=30), paper_bgcolor='white', plot_bgcolor='white')
    if out_prefix is None:
        out_prefix = f'{cancer_type}_method-{method_name}_top{top_k_drugs}_targets{n_targets}'
    out_pdf = os.path.join(out_dir, f'{out_prefix}.pdf')
    fig.write_html(os.path.join(out_dir, f'{out_prefix}.html'), include_plotlyjs=True)
    if EXPORT_SANKEY_PDF:
        try:
            fig.write_image(out_pdf, format='pdf', scale=3)
        except (ImportError, ValueError, RuntimeError) as error:
            print(f'Static Sankey PDF export unavailable: {error}. The interactive HTML was saved.')
    if os.path.isfile(out_pdf):
        print('Saved:', out_pdf)
    out_map = os.path.join(out_dir, f'{out_prefix}_target_mapping.tsv')
    target_map_df.to_csv(out_map, sep='\t', index=False)
    print('Saved:', out_map)
    fig.show()
    return (mat_top, target_map_df)


In [ ]:
if RUN_DRUG_ANALYSIS and RUN_DRUG_SANKEY:
    mat_top, target_map = plot_target_drug_sankey_topk(
        mat=drug_matrix, cancer_type="LUNG", method_name="max", top_k_drugs=10,
        n_targets=4, out_dir=str(DRUG_OUTPUT_DIR), scale_link_values=6.0)
    mat_top.to_csv(DRUG_OUTPUT_DIR / "top10_target_affinities.csv")


## 8. Save run settings

Record paths, scoring parameters, concept checksums, and software versions with the results.


In [ ]:
configuration = dict(drug_ranking=dict(method="max", target_genes=TARGET_GENES, input_dir=str(DRUG_INPUT_DIR),
    input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in DRUG_INPUT_DIR.iterdir() if p.is_file() and p.name != "manifest.json"}),
    python=platform.python_version(),
    packages={p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "scanpy", "anndata", "matplotlib", "lifelines", "statsmodels"]},
    inputs=dict(atlas=str(ATLAS_PATH), tcga=str(TCGA_PATH), scissor_labels=str(SCISSOR_LABEL_PATH), pathways=str(GMT_PATH)),
    concept_sha256=hashlib.sha256(CONCEPT_PATH.read_bytes()).hexdigest(),
    top_k=dict(atlas=ATLAS_TOP_K, tcga=TCGA_TOP_K, scissor=SCISSOR_TOP_K),
    enabled=dict(atlas=RUN_ATLAS, atlas_plots=RUN_ATLAS_PLOTS, interpretability=RUN_INTERPRETABILITY,
                 tcga=RUN_TCGA, stage=RUN_STAGE_PLOTS, survival=RUN_SURVIVAL, scissor=RUN_SCISSOR))
configuration["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(configuration, indent=2))
print("Outputs:", OUTPUT_DIR)
